In [16]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings, OllamaLLM
from langchain_chroma import Chroma
from langchain_classic.chains.retrieval_qa.base import RetrievalQA
from langchain_core.prompts import PromptTemplate

import os
import json

Approach 1: Using Langchain with Ollama and Chroma

In [2]:
#Pull the embedding model "nomic-embed-text" and LLM model "llama3.2" or other models of your choice
#!ollama pull nomic-embed-text
#!ollama pull llama3.2

In [3]:
# Load PDF
loader = PyPDFLoader("Drivers-Handbook.pdf")
documents = loader.load()

In [4]:
# Split
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)
chunks = text_splitter.split_documents(documents)

In [5]:
# Embeddings
embeddings = OllamaEmbeddings(model="nomic-embed-text")

In [6]:
# Vector store 
vectorstore_directory="./chroma_db_pdf"

if os.path.exists(vectorstore_directory) and os.listdir(vectorstore_directory):
    print("Loading existing vector store.....")
    vectorstore = Chroma(persist_directory=vectorstore_directory, embedding_function = embeddings)
else:
    print("Building vector store.....")
    vectorstore = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        persist_directory=vectorstore_directory
    )

Loading existing vector store.....


In [9]:
# LLM
llm = OllamaLLM(model="llama3.2", temperature=0.3)

In [10]:
"""Create the QA chain with custom prompt"""
        
# Custom prompt template
template = """You are an expert on German traffic rules. Use the following context to answer the question.
If you don't know the answer, just say you don't know. Don't make up an answer.

Context: {context}

Question: {question}

Answer: """

prompt = PromptTemplate(
    template=template,
    input_variables=["context", "question"]
)

In [11]:
# QA Chain
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=vectorstore.as_retriever(search_kwargs={"k": 3}),
    return_source_documents=True,
    chain_type_kwargs={"prompt": prompt}
)

In [12]:
# Query
result = qa_chain.invoke({"query": "What are the criminal offences of a driver in Germany"})
print(result['result'])

According to German traffic rules, the criminal offenses of a driver in Germany include:

1. Operating a POV with a blood alcohol content (BAC) above the legal limit.
2. Operating a class of vehicle other than the class for which licensed.
3. Owning or operating an unregistered or uninsured POV.

Additionally, there are also specific offenses related to driving under the influence of drugs and alcohol:

1. Driving while intoxicated (with a BAC between 0.05 grams per 100 milliliters and 0.79 grams per 100 milliliters).
2. Failing to wear a seatbelt or requiring passengers to wear a seatbelt or restraining device while riding in a POV (third and subsequent offenses).

Note that the specific penalties for these offenses, such as fines and imprisonment, may vary depending on the severity of the offense and other factors.


In [13]:
# Query
result = qa_chain.invoke({"query": "what is the speed limit in autobahn for cars?"})
print(result['result'])

The recommended maximum speed limit on autobahns for single vehicles with up to 3.5 tons of authorized loaded weight is 130 kph (62 mph).


In [17]:
# Query
result = qa_chain.invoke({"query": "Rules in europe for driving during fog and smoke"})

In [18]:
print(json.dumps(result, indent=2))

TypeError: Object of type Document is not JSON serializable